<a href="https://colab.research.google.com/github/BeachBall2024/GEO-spa-algo/blob/main/submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Workflow and Introduction
Study:
1. Load data and basic functions
- Lea
2. preprocess Flickr data for typos, upper-lower case
3. construct sound wheel
- Julia
4. divide streets into segments (name, then again for a max length)
- Muriel
5. create Buffer around streets
- Guillermo
6. check which points lay inside buffer and match them to sound categories, count

In [ ]:
#Import gihub repo libraries
!git clone https://github.com/BeachBall2024/GEO-spa-algo.git
%cd GEO-spa-algo

2. Import Python packages

In [ ]:
#importing important packages

3. Import basic geometric functions
(Functions are dividet in different classes based on shape)

- Point
- Line
- Polygon

In [ ]:
#POINT
class Point():
    # initialise
    def __init__(self, x=None, y=None):
        self.x = x
        self.y = y

    # representation
    def __repr__(self):
        return f'Point(x={self.x}, y={self.y})'

    # Test for equality between Points
    def __eq__(self, other):
        if not isinstance(other, Point): #test if point is within the class
            # don't attempt to compare against unrelated types
            return NotImplemented
        return self.x == other.x and self.y == other.y

    # We need this method so that the class will behave sensibly in sets and dictionaries
    def __hash__(self):
        return hash((self.x, self.y))

class PointGroup():
    # initialise
    def __init__(self, data=None, xcol=None, ycol=None):
        self.points = []
        self.size = len(data)
        for d in data:
            self.points.append(Point(d[xcol], d[ycol]))

    # representation
    def __repr__(self):
        return f'PointGroup containing {self.size} points'

    # create index of points in group for referencing
    def __getitem__(self, key):
        return self.points[key]

In [ ]:
#LINE
class Segment():
    # initialise
    def __init__(self, p0, p1, sid=None):
        self.start = p0
        self.end = p1
        self.sid = sid

    # representation
    def __repr__(self):
        return f'Segment with start {self.start} and end {self.end}.'

    # test if segment is identical to another segment (using Point method isEqual)
    def isIdentical(self, other):
        if (self.start.isEqual(other.start)
            or self.start.isEqual(other.end)) and (self.end.isEqual(other.end)
                                                   or self.end.isEqual(other.start)):
            return True
        else:
            return False

    # determine if intersects with another segment (using Point method sideLine)
    # - should we incorporate testing for identical segments and non-zero lengths?
    def intersects(self, other):
        # create bounding boxes for each segment, using Bbox class
        self_bbox = Bbox(self)
        other_bbox = Bbox(other)

        # test if bounding boxes overlap (using Bbox method testOverlap)
        bbox_overlap = self_bbox.testOverlap(other_bbox)

        # if the two bboxes do not overlap, there can be no intersection
        if bbox_overlap == False:
            return False

        else:
            return True

    def length(self):
        #calculate the distance of the segment

In [ ]:
#POLYGON
# Polygon class for polygons, assumes initial data is in a spatially sorted order
class PointGroup:
    # initialise
    def __init__(self, data=None, xcol=None, ycol=None):
        self.points = []
        self.size = len(data)
        for d in data:
            self.points.append(Point(d[xcol], d[ycol]))

    # representation
    def __repr__(self):
        return f'Polygon PointGroup containing {self.size} points'

    # test if polygon is closed: first and last point should be identical
    def isClosed(self):
        start = self.points[0]
        end = self.points[-1]
        return start == end

    def removeDuplicates(self):
        oldn = len(self.points)
        self.points = list(dict.fromkeys(self.points)) # Get rid of the duplicates
        self.points.append(self.points[0]) # Our polygon must have one duplicate - we put it back now
        n = len(self.points)
        self.size = n   # see how the absence/presence of this line makes changes
        print(f'The old polygon had {oldn} points, now we only have {n}.')

# 4. Data Loading

  We load three datasets:
- **Flickr Images**
- **Sound Wheel**
- **Streets of Switzerland**


In [ ]:
#import data

# 5. Preprocessing of Flickr Images



In [ ]:
#preprocess of Flickr Images

6. Division of Streets into Segments

In [ ]:
import math
import fiona
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import linemerge

# ── Settings ──────────────────────────────────────────────────────────────────
GDB_FILE             = "amtliches-strassenverzeichnis_ch_2056.gdb"
OUT_SEGMENTS         = "zurich_street_segments.csv"
MAX_SEGMENT_LENGTH_M = 500
# ──────────────────────────────────────────────────────────────────────────────


def cut_line_into_segments(line, max_length):
    """
    Cut a Shapely LineString into pieces no longer than max_length metres.
    Returns a list of LineStrings.

    How it works:
      - Walk along the line in steps of max_length
      - At each step, extract the sub-line between start and end distance
      - Stop when we've covered the full length
    """
    total = line.length

    if total <= max_length:
        return [line]   # short enough already, nothing to cut

    segments = []
    start_dist = 0.0

    while start_dist < total:
        end_dist = min(start_dist + max_length, total)
        segments.append(_extract_substring(line, start_dist, end_dist))
        start_dist = end_dist

    return segments


def _extract_substring(line, start_dist, end_dist):
    """
    Pull out the part of a line between two distances (in metres along the line).
    Interpolates exact start/end points so segments fit together perfectly.
    """
    coords = []
    current = 0.0
    started = False

    pts = list(line.coords)
    for i in range(len(pts) - 1):
        p1, p2 = pts[i], pts[i + 1]
        seg_len = math.dist(p1, p2)

        # Have we reached the start of our desired segment?
        if not started and current + seg_len >= start_dist:
            t = (start_dist - current) / seg_len
            coords.append((p1[0] + t * (p2[0] - p1[0]),
                           p1[1] + t * (p2[1] - p1[1])))
            started = True

        if started:
            if current + seg_len >= end_dist:
                # We've reached the end → interpolate exact endpoint and stop
                t = (end_dist - current) / seg_len
                coords.append((p1[0] + t * (p2[0] - p1[0]),
                               p1[1] + t * (p2[1] - p1[1])))
                break
            else:
                coords.append(p2)

        current += seg_len

    return LineString(coords)


# ── Step 1: Load the three GDB layers we need ─────────────────────────────────
print("Loading GDB ...")

lin_gdf = gpd.read_file(GDB_FILE, layer="PURE_LIN")   # actual street geometries

with fiona.open(GDB_FILE, layer="PURE_STR") as f:
    str_df = pd.DataFrame([feat["properties"] for feat in f])   # commune / canton

with fiona.open(GDB_FILE, layer="PURE_STN") as f:
    stn_df = pd.DataFrame([feat["properties"] for feat in f])   # street names

print(f"  {len(lin_gdf):,} street geometries loaded.")

# ── Step 2: Join name and commune info onto the geometry ──────────────────────
streets = (
    lin_gdf
    .merge(str_df[["STR_ESID", "COM_NAME"]], on="STR_ESID")
    .merge(stn_df[["STR_ESID", "STN_TEXT"]], on="STR_ESID")
)

# ── Step 3: Filter to Zürich city ─────────────────────────────────────────────
zurich = streets[streets["COM_NAME"] == "Zürich"].copy()
print(f"  {len(zurich):,} streets in Zürich.")

# ── Step 4: Split each street into ≤500 m segments ───────────────────────────
print("Splitting into segments ...")

rows = []

for _, row in zurich.iterrows():
    geom = row.geometry

    # Streets are stored as MultiLineString (multiple sub-lines).
    # We try to merge them into one continuous LineString first.
    if geom.geom_type == "MultiLineString":
        merged = linemerge(geom)
        if merged.geom_type == "MultiLineString":
            # Still disconnected → flatten all coordinates into one line
            merged = LineString([c for part in merged.geoms for c in part.coords])
    else:
        merged = geom

    segments = cut_line_into_segments(merged, MAX_SEGMENT_LENGTH_M)

    for seg_id, seg in enumerate(segments, start=1):
        coords = list(seg.coords)
        rows.append({
            "street_name":    row["STN_TEXT"],
            "segment_id":     seg_id,
            "total_segments": len(segments),
            "length_m":       round(seg.length, 1),
            "start_easting":  round(coords[0][0], 1),
            "start_northing": round(coords[0][1], 1),
            "end_easting":    round(coords[-1][0], 1),
            "end_northing":   round(coords[-1][1], 1),
        })

# ── Step 5: Save ──────────────────────────────────────────────────────────────
result = pd.DataFrame(rows)
result.to_csv(OUT_SEGMENTS, index=False, encoding="utf-8-sig")

print(f"\nDone! {len(result):,} segments saved to '{OUT_SEGMENTS}'.")
print(f"Streets split: {result[result['total_segments'] > 1]['street_name'].nunique()}")
print(f"\nTop 5 longest streets (total length):")
print(result.groupby("street_name")["length_m"].sum().sort_values(ascending=False).head(5).to_string())

# 7. Creation of Buffer

In [ ]:
# create Buffer

8. Point in Polygon


In [ ]:
#point in buffer testing

9. Visualization

In [ ]:
#sound map